# Team 5: How can we affect the pollutants?

## Introduction

To-Do:
- Write a summary of the section

### Why

To-Do:
- Explain what those pollutants are / where they come from
- What are the effects of the pollutants on the body
- Why do we focus on these pollutants and not co2 etc.

### Model Types

To-Do:
- Give a reason why there are 3 models and how the work together.

### European Model
WIP
- Apply Data to get delta PM / NO2 values
- WHO and Eurostat

### Local Model

To-Do:
- Describe why it has to be used.
- Describe the approach.
- Describe the sources

### System Model

To-Do:
- Describe why it has to be used.
- Describe the approach.
- Describe the sources

## Coding

### European Model

#### Import Libraries

In [ ]:
import pandas as pd
import xgboost as xgb
from functools import reduce
import numpy as np 
from numpy import polyfit, polyval
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
import os

#### Load Data

Loading all the data and put them into a standard format.

In [ ]:
path = "data/"
df_bus_age = pd.read_csv(path+ 'road_eqs_busage_linear_2_0.csv')
df_bus_motor = pd.read_csv(path+ 'road_eqs_busmot_linear_2_0.csv')
df_car_age = pd.read_csv(path+ 'road_eqs_carage_linear_2_0.csv')
df_car_motor = pd.read_csv(path+ 'road_eqs_carmot_linear_2_0.csv')
df_gdp_per_capita = pd.read_csv(path+ 'sdg_08_10_linear_2_0.csv')
df_pop = pd.read_csv(path+ 'tps00001_population.csv') # Only since 2014
df_pop_density = pd.read_csv(path+ 'tps00003_pop_density.csv') # Only since 2012
df_pop_age = pd.read_csv(path+ 'tps00010_pop_age.csv') # Only since 2013
df_modal_share = pd.read_csv(path+ 'tran_hv_psmod_linear_2_0.csv')
df_aq = pd.read_csv(path+ 'who_ambient_air_quality_database_version_2024_(v6.1)(Update 2024 (V6.csv')

Create a function to clean up all the different country names for the same country.

In [ ]:
def standardize_country_name(name):
    replacements = {
        "Netherlands (Kingdom of the)": "Netherlands",
        "T√ºrkiye": "Türkiye",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "Metropolitan France": "France",
        "Russian Federation": "Russia"
    }
    return replacements.get(name, name)

Extract all the wanted data and standardize the formatting of the DataFrame.

In [ ]:
def pivot_eurostat(df, pivot_colum, time = "TIME_PERIOD", location = "Geopolitical entity (reporting)", value = "OBS_VALUE"):
    df_filtered = df[df[time] >= 2010]
    df_pivot = df_filtered.pivot_table(
    index=[location, time],
    columns=pivot_colum,
    values=value
    )
    df_pivot = df_pivot.reset_index()
    df_pivot[location] = df_pivot[location].apply(standardize_country_name)
    return df_pivot

In [ ]:
df_bus_age_clean = pivot_eurostat(df = df_bus_age, pivot_colum = 'age')
df_bus_age_clean.drop('Y_GE10', axis=1, inplace=True)
df_bus_age_clean.columns = ['Country', 'Year', 'Bus_age_Total', 'Bus_age_10-20', 'Bus_age_2-5', 'Bus_age_5-10','Bus_age_GT_20','Bus_age_LT_2']
df_bus_age_clean['Country'] = df_bus_age_clean['Country'].apply(standardize_country_name)
df_bus_age_clean

In [ ]:
df_bus_motor_clean = pivot_eurostat(df = df_bus_motor, pivot_colum = 'Motor energy')
df_bus_motor_clean = df_bus_motor_clean[['Geopolitical entity (reporting)', 'TIME_PERIOD', 'Diesel', 'Petroleum products', 'Liquefied petroleum gases (LPG)', 'Natural gas', 'Electricity']]
df_bus_motor_clean.columns = ['Country', 'Year', 'Bus_Diesel', 'Bus_Petroleum', 'Bus_LPG', 'Bus_Natural_Gas', 'Bus_Electricity']
df_bus_motor_clean['Country'] = df_bus_motor_clean['Country'].apply(standardize_country_name)   
df_bus_motor_clean

In [ ]:
df_car_age_clean = pivot_eurostat(df = df_bus_age, pivot_colum = 'age').drop('Y_GE10', axis=1)
df_car_age_clean.columns = ['Country', 'Year', 'Car_age_Total', 'Car_age_10-20', 'Car_age_2-5', 'Car_age_5-10','Car_age_GT_20','Car_age_LT_2']
df_car_age_clean['Country'] = df_car_age_clean['Country'].apply(standardize_country_name)
df_car_age_clean


In [ ]:
df_car_motor_clean = pivot_eurostat(df = df_car_motor, pivot_colum = 'Motor energy')
df_car_motor_clean.columns = ['Country', 'Year', 'Car_Diesel', 'Car_Petroleum', 'Car_motor_Total']
df_car_motor_clean['Car_Renewable'] = df_car_motor_clean['Car_motor_Total'] - df_car_motor_clean['Car_Diesel'] - df_car_motor_clean['Car_Petroleum']
df_car_motor_clean['Country'] = df_car_motor_clean['Country'].apply(standardize_country_name)
df_car_motor_clean

In [ ]:
df_gdp_per_capita_clean = df_gdp_per_capita[df_gdp_per_capita['Unit of measure'] == 'Chain linked volumes (2020), euro per capita']
df_gdp_per_capita_clean = df_gdp_per_capita_clean[['Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE']]
df_gdp_per_capita_clean.columns = ['Country', 'Year', 'GDP per capita']
df_gdp_per_capita_clean = df_gdp_per_capita_clean[df_gdp_per_capita_clean['Year'] >= 2010]
df_gdp_per_capita_clean['Country'] = df_gdp_per_capita_clean['Country'].apply(standardize_country_name)
df_gdp_per_capita_clean

In [ ]:
df_pop_clean = df_pop[['Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE']]
df_pop_clean.columns = ['Country', 'Year', 'Population']
df_pop_clean['Country'] = df_pop_clean['Country'].apply(standardize_country_name)
df_pop_clean

In [ ]:
df_pop_density_clean = df_pop_density[['Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE']]
df_pop_density_clean.columns = ['Country', 'Year', 'Population_density']
df_pop_density_clean['Country'] = df_pop_density_clean['Country'].apply(standardize_country_name)
df_pop_density_clean

In [ ]:
df_pop_age_clean = pivot_eurostat(df = df_pop_age, pivot_colum = 'indic_de')
df_pop_age_clean.columns = ['Country', 'Year', 'Pop_age_0-14', 'Pop_age_15-24', 'Pop_age_25-49', 'Pop_age_50-64', 'Pop_age_65-79', 'Pop_age_GT_80']
df_pop_age_clean['Country'] = df_pop_age_clean['Country'].apply(standardize_country_name)
df_pop_age_clean

In [ ]:
df_modal_share_clean = pivot_eurostat(df = df_modal_share, pivot_colum = 'vehicle').drop(['TRN_BUS_TOT_AVD'], axis=1)
df_modal_share_clean.columns = ['Country', 'Year', 'Modal_Share_Bus', 'Modal_Share_Car', 'Modal_Share_Train']
df_modal_share_clean['Country'] = df_modal_share_clean['Country'].apply(standardize_country_name)
df_modal_share_clean

In [ ]:
df_aq_eu = df_aq[df_aq['who_region'] == '4_Eur']
df_aq_clean = df_aq_eu[['country_name', 'year', 'pm10_concentration', 'pm25_concentration', 'no2_concentration']]
df_aq_clean = df_aq_clean[df_aq_clean['year'] >= 2010]
df_aq_clean.columns = ['Country', 'Year', 'pm10_concentration', 'pm25_concentration', 'no2_concentration']
df_aq_clean_mean = df_aq_clean.groupby(["Country", "Year"])[['pm10_concentration', 'pm25_concentration', 'no2_concentration']].mean()
df_aq_clean_mean = df_aq_clean_mean.reset_index()
df_aq_clean_mean['Country'] = df_aq_clean_mean['Country'].apply(standardize_country_name)
df_aq_clean_mean

#### Join the Datasets and clean important NA values

In [ ]:
# List of all cleaned dataframes to join
dfs = [
    df_bus_age_clean,
    df_bus_motor_clean,
    df_car_age_clean,
    df_car_motor_clean,
    df_gdp_per_capita_clean,
    df_pop_clean,
    df_pop_density_clean,
    df_pop_age_clean,
    df_modal_share_clean,
    df_aq_clean_mean
]

# Merge all dataframes on 'Country' and 'Year'
df_all = reduce(lambda left, right: pd.merge(left, right, on=['Country', 'Year'], how='outer'), dfs)
df_all

In [ ]:
df_all_clean = df_all.dropna(subset=['pm10_concentration', 'pm25_concentration', 'no2_concentration'], how='all')
df_all_clean = df_all_clean.dropna(subset=['Modal_Share_Bus', 'Modal_Share_Car', 'Modal_Share_Train'], how='all')
df_all_clean


#### Calculations

Normalize the values to reduce impact for the population size of the country.
For this, where appropriate, the values were changed to percentages.

In [ ]:
df_all_clean_calc = df_all_clean.copy()
# Car Motor
df_all_clean_calc['Car_Renewable'] = df_all_clean_calc['Car_Renewable'] / df_all_clean_calc['Car_motor_Total'] * 100
df_all_clean_calc['Car_Petroleum'] = df_all_clean_calc['Car_Petroleum'] / df_all_clean_calc['Car_motor_Total'] * 100
df_all_clean_calc['Car_Diesel'] = df_all_clean_calc['Car_Diesel'] / df_all_clean_calc['Car_motor_Total'] * 100

# Car Age
df_all_clean_calc["Car_age_LT_2"] = df_all_clean_calc["Car_age_LT_2"] / df_all_clean_calc["Car_age_Total"] * 100
df_all_clean_calc["Car_age_2-5"] = df_all_clean_calc["Car_age_2-5"] / df_all_clean_calc["Car_age_Total"] * 100
df_all_clean_calc["Car_age_5-10"] = df_all_clean_calc["Car_age_5-10"] / df_all_clean_calc["Car_age_Total"] * 100
df_all_clean_calc["Car_age_10-20"] = df_all_clean_calc["Car_age_10-20"] / df_all_clean_calc["Car_age_Total"] * 100
df_all_clean_calc["Car_age_GT_20"] = df_all_clean_calc["Car_age_GT_20"] / df_all_clean_calc["Car_age_Total"] * 100

# Bus Motor
df_all_clean_calc['Bus_motor_Total'] = df_all_clean_calc['Bus_Diesel'] + df_all_clean_calc['Bus_Petroleum'] + df_all_clean_calc['Bus_LPG'] + df_all_clean_calc['Bus_Natural_Gas'] + df_all_clean_calc['Bus_Electricity']
df_all_clean_calc['Bus_LPG'] = df_all_clean_calc['Bus_LPG'] / df_all_clean_calc['Bus_motor_Total'] * 100
df_all_clean_calc['Bus_Petroleum'] = df_all_clean_calc['Bus_Petroleum'] / df_all_clean_calc['Bus_motor_Total'] * 100
df_all_clean_calc['Bus_Diesel'] = df_all_clean_calc['Bus_Diesel'] / df_all_clean_calc['Bus_motor_Total'] * 100
df_all_clean_calc['Bus_Natural_Gas'] = df_all_clean_calc['Bus_Natural_Gas'] / df_all_clean_calc['Bus_motor_Total'] * 100
df_all_clean_calc['Bus_Electricity'] = df_all_clean_calc['Bus_Electricity'] / df_all_clean_calc['Bus_motor_Total'] * 100

# Bus Age
df_all_clean_calc["Bus_age_LT_2"] = df_all_clean_calc["Bus_age_LT_2"] / df_all_clean_calc["Bus_age_Total"] * 100
df_all_clean_calc["Bus_age_2-5"] = df_all_clean_calc["Bus_age_2-5"] / df_all_clean_calc["Bus_age_Total"] * 100
df_all_clean_calc["Bus_age_5-10"] = df_all_clean_calc["Bus_age_5-10"] / df_all_clean_calc["Bus_age_Total"] * 100
df_all_clean_calc["Bus_age_10-20"] = df_all_clean_calc["Bus_age_10-20"] / df_all_clean_calc["Bus_age_Total"] * 100
df_all_clean_calc["Bus_age_GT_20"] = df_all_clean_calc["Bus_age_GT_20"] / df_all_clean_calc["Bus_age_Total"] * 100

# Total transport per capita.
df_all_clean_calc['Car_age_Total'] = df_all_clean_calc['Car_age_Total'] / df_all_clean_calc['Population'] * 100
df_all_clean_calc['Bus_age_Total'] = df_all_clean_calc['Bus_age_Total'] / df_all_clean_calc['Population'] * 100


In [ ]:
df_all_clean_calc.describe()

#### Analyzing the corelation of the pollutants and the data values.

In [ ]:
def plot_pollutant_vs_x(df, pollutant, split):
    print(f"Plotting {pollutant}")
    fig_height = (len(split)//3)+1
    if len(split) <= 3:
        fig, axes = plt.subplots(1, len(split), figsize=(15, 4), sharey=True)
    elif len(split) % 3 == 0:
        fig, axes = plt.subplots(len(split)//3, 3, figsize=(6*(len(split)//3), (6*(len(split)//3))*2/3), sharey=True)
    else:
        fig, axes = plt.subplots(fig_height, 3, figsize=(6*fig_height, (6*fig_height)*2/3), sharey=True)
    corrs = {}

    axes_flat = axes.flatten() if hasattr(axes, "flatten") else np.array([axes])

    for i, type_split in enumerate(split):
        mask = ~df[type_split].isna() & ~df[pollutant].isna()
        sc = axes_flat[i].scatter(df[type_split][mask], df[pollutant][mask], c=df["Year"][mask], cmap="viridis", alpha=0.6)
        coef = polyfit(df[type_split][mask], df[pollutant][mask], 1)
        x = np.linspace(df[type_split].min(), df[type_split].max(), 100)
        axes_flat[i].plot(x, polyval(coef, x), color="red", label="Regression line")
        axes_flat[i].set_xlabel(type_split)
        axes_flat[i].set_title(f"{type_split} vs {pollutant.upper()}")
        axes_flat[i].legend()
        corr = df[type_split][mask].corr(df[pollutant][mask])
        corrs[type_split] = corr

    plt.tight_layout()
    plt.show()
    return corrs

Initialize the storage of the correlations for each datasets.

In [ ]:
correlation_results_propultion = {}
correlation_results_type_age = {}
correlation_results_modal_share = {}
correlation_results_Total_transport = {}
correlation_results_POP_Age = {}
correlation_results_GDP = {}